# Balanced Binary Dataset for XLM-RoBERTa-large
Questo notebook crea un dataset bilanciato per la classificazione binaria AI / NON AI, unendo dati xlsx strutturati, testi sparsi (TXT) categorizzati dalla cartella, e colmando eventuali deficit con campioni da `bert_binary.csv`.

In [19]:
import pandas as pd
import glob
import os
import random

%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


## 1. Caricamento dati strutturati (XLSX)
Lettura dei file `train_*.xlsx`.

In [20]:
excel_files = glob.glob('../../data/binary/train_*.xlsx')
dfs = []
for f in excel_files:
    df = pd.read_excel(f)
    dfs.append(df)

if dfs:
    df_excel = pd.concat(dfs, ignore_index=True)
else:
    df_excel = pd.DataFrame(columns=['text', 'label'])

# Rinominiamo le colonne in modo standard
df_excel = df_excel.rename(columns={'Descrizione': 'text', 'Label': 'label'})

# Normalizziamo le etichette
if 'label' in df_excel.columns:
    df_excel['label'] = df_excel['label'].replace({
        1: 'AI', 0: 'NON AI', 
        '1': 'AI', '0': 'NON AI',
        'ai': 'AI', 'non_ai': 'NON AI'
    })

print("Distribuzione Classi (XLSX):")
if 'label' in df_excel.columns:
    print(df_excel['label'].value_counts(dropna=False))

Distribuzione Classi (XLSX):
label
NON AI    6553
AI        1578
Name: count, dtype: int64


ai_txt_files = glob.glob('../../data/binary/ai/*.txt')
non_ai_txt_files = glob.glob('../../data/binary/non_ai/*.txt')

txt_data = []
for f in ai_txt_files:
    with open(f, 'r', encoding='utf-8') as file:
        txt_data.append({'text': file.read().strip(), 'label': 'AI'})

for f in non_ai_txt_files:
    with open(f, 'r', encoding='utf-8') as file:
        txt_data.append({'text': file.read().strip(), 'label': 'NON AI'})

# Forziamo le colonne in caso di lista vuota
df_txt = pd.DataFrame(txt_data, columns=['text', 'label'])
print("Distribuzione Classi (TXT):")
if not df_txt.empty:
    print(df_txt['label'].value_counts())

In [21]:
ai_txt_files = glob.glob('../../data/binary/ai/*.txt')
non_ai_txt_files = glob.glob('../../data/binary/non_ai/*.txt')

txt_data = []
for f in ai_txt_files:
    with open(f, 'r', encoding='utf-8') as file:
        txt_data.append({'text': file.read().strip(), 'label': 'AI'})

for f in non_ai_txt_files:
    with open(f, 'r', encoding='utf-8') as file:
        txt_data.append({'text': file.read().strip(), 'label': 'NON AI'})

df_txt = pd.DataFrame(txt_data)
print("Distribuzione Classi (TXT):")
if not df_txt.empty:
    print(df_txt['label'].value_counts())

Distribuzione Classi (TXT):
label
NON AI    1175
AI         581
Name: count, dtype: int64


# Per sicurezza, ci assicuriamo che le colonne necessarie esistano, 
# se non hai rieseguito le celle superiori dopo le modifiche.
for col, target in [('Descrizione', 'text'), ('Label', 'label'), ('testo', 'text')]:
    if col in df_excel.columns and target not in df_excel.columns:
        df_excel = df_excel.rename(columns={col: target})

if 'label' in df_excel.columns:
    df_excel['label'] = df_excel['label'].replace({
        1: 'AI', 0: 'NON AI', 
        '1': 'AI', '0': 'NON AI',
        'ai': 'AI', 'non_ai': 'NON AI'
    })

# Uniamo solo se le colonne esistono (altrimenti df vuota)
if 'text' in df_excel.columns and 'label' in df_excel.columns:
    excel_subset = df_excel[['text', 'label']]
else:
    excel_subset = pd.DataFrame(columns=['text', 'label'])

df_combined = pd.concat([excel_subset, df_txt[['text', 'label']]], ignore_index=True)
df_combined = df_combined.dropna(subset=['text', 'label'])

current_counts = df_combined['label'].value_counts() if 'label' in df_combined.columns else pd.Series(dtype=int)
print("Distribuzione attuale parziale:")
print(current_counts)

# Caricamento bert_binary.csv
df_bert = pd.read_csv('../../data/training/bert_binary.csv')

# Uniformiamo la colonna del testo
for col in ['testo', 'Descrizione']:
    if col in df_bert.columns:
        df_bert = df_bert.rename(columns={col: 'text'}) 

# Uniformiamo l'etichetta
for col in ['Label']:
    if col in df_bert.columns:
        df_bert = df_bert.rename(columns={col: 'label'})

if 'label' in df_bert.columns:
    df_bert['label'] = df_bert['label'].replace({
        1: 'AI', 0: 'NON AI', 
        '1': 'AI', '0': 'NON AI',
        'ai': 'AI', 'non_ai': 'NON AI'
    })

ai_count = current_counts.get('AI', 0)
non_ai_count = current_counts.get('NON AI', 0)
diff = ai_count - non_ai_count

sampled_dfs = [df_combined]

if diff > 0:
    # Servono più campioni NON AI per equiparare
    df_bert_target = df_bert[df_bert['label'] == 'NON AI'].dropna(subset=['text'])
    sample_size = min(diff, len(df_bert_target))
    if sample_size > 0:
        sampled_dfs.append(df_bert_target.sample(n=sample_size, random_state=42)[['text', 'label']])
        print(f"Aggiunto {sample_size} campione/i NON AI da bert_binary.csv")
elif diff < 0:
    # Servono più campioni AI per equiparare
    df_bert_target = df_bert[df_bert['label'] == 'AI'].dropna(subset=['text'])
    sample_size = min(abs(diff), len(df_bert_target))
    if sample_size > 0:
        sampled_dfs.append(df_bert_target.sample(n=sample_size, random_state=42)[['text', 'label']])
        print(f"Aggiunto {sample_size} campione/i AI da bert_binary.csv")
else:
    print("Classi già perfettament bilanciate o dati del file non caricati correttamente.")

df_final = pd.concat(sampled_dfs, ignore_index=True)
print("\nDistribuzione FINALE prima dello shuffle:")
if 'label' in df_final.columns:
    print(df_final['label'].value_counts())
else:
    print("Errore: la colonna 'label' non è presente nel dataset finale.")

In [22]:
# df_excel ha già le colonne 'text' e 'label'
df_combined = pd.concat([df_excel[['text', 'label']], df_txt[['text', 'label']]], ignore_index=True)
df_combined = df_combined.dropna(subset=['text', 'label'])

current_counts = df_combined['label'].value_counts() if 'label' in df_combined.columns else pd.Series(dtype=int)
print("Distribuzione attuale parziale:")
print(current_counts)

# Caricamento bert_binary.csv
df_bert = pd.read_csv('../../data/training/bert_binary.csv')

# Uniformiamo la colonna del testo se si chiama in modo diverso
for col in ['testo', 'Descrizione', 'description']:
    if col in df_bert.columns:
        df_bert = df_bert.rename(columns={col: 'text'})

# Uniformiamo l'etichetta
for col in ['Label', 'class', 'CLASSIFICAZIONE']:
    if col in df_bert.columns:
        df_bert = df_bert.rename(columns={col: 'label'})

if 'label' in df_bert.columns:
    df_bert['label'] = df_bert['label'].replace({
        1: 'AI', 0: 'NON AI', 
        '1': 'AI', '0': 'NON AI',
        'ai': 'AI', 'non_ai': 'NON AI'
    })

ai_count = current_counts.get('AI', 0)
non_ai_count = current_counts.get('NON AI', 0)
diff = ai_count - non_ai_count

sampled_dfs = [df_combined]

if 'label' in df_bert.columns:
    if diff > 0:
        # Servono più campioni NON AI per equiparare
        df_bert_target = df_bert[df_bert['label'] == 'NON AI'].dropna(subset=['text'])
        sample_size = min(diff, len(df_bert_target))
        if sample_size > 0:
            sampled_dfs.append(df_bert_target.sample(n=sample_size, random_state=42)[['text', 'label']])
            print(f"Aggiunto {sample_size} campione/i NON AI da bert_binary.csv")
    elif diff < 0:
        # Servono più campioni AI per equiparare
        df_bert_target = df_bert[df_bert['label'] == 'AI'].dropna(subset=['text'])
        sample_size = min(abs(diff), len(df_bert_target))
        if sample_size > 0:
            sampled_dfs.append(df_bert_target.sample(n=sample_size, random_state=42)[['text', 'label']])
            print(f"Aggiunto {sample_size} campione/i AI da bert_binary.csv")
    else:
        print("Classi già perfettament bilanciate.")
else:
    print("ERRORE: impossile trovare colonna 'label' nel dataframe df_bert_target.")

df_final = pd.concat(sampled_dfs, ignore_index=True)
print("\nDistribuzione FINALE prima dello shuffle:")
print(df_final['label'].value_counts())

Distribuzione attuale parziale:
label
NON AI    7723
AI        2159
Name: count, dtype: int64
Aggiunto 5564 campione/i AI da bert_binary.csv

Distribuzione FINALE prima dello shuffle:
label
NON AI    7723
AI        7723
Name: count, dtype: int64


## 4. Shuffling e Salvataggio Dataset

In [23]:
# Shuffle dei dati per la validazione di train/test/split successiva
df_final = df_final.sample(frac=1.0, random_state=42).reset_index(drop=True)

output_path = '../../data/training/bert_bin_unbiased.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_final.to_csv(output_path, index=False)
print(f"Dataset randomizzato e salvato con successo in {output_path} (totale righe: {len(df_final)})")

Dataset randomizzato e salvato con successo in ../../data/training/bert_bin_unbiased.csv (totale righe: 15446)
